# zi2zi-JiT · 字体训练 AutoDL Notebook（一键版 · 默认按 V100 32G / 6 核适配）

基于 **zi2zi-JiT**（ICW fork），专为 **AutoDL 租卡 Jupyter** 设计。所有参数集中在 **Cell 0**，其余代码 Cell 无需修改，从前往后依次运行即可。

**三步上手：**

1. 上传素材：预训练模型 → `models/`；目标字体 + 参照字体（.ttf/.otf）→ `fonts/`
2. 改 **Cell 0** 里标了 🔴 **必改** 的几个参数（目标字体文件名、模型配套）
3. 依次运行 Cell 0 → 6，产物在 `outputs/<字体>/`，zip 在 `outputs/`

**目录约定：**

| 目录 | 内容 |
|---|---|
| `fonts/` | 目标字体（要学的）+ 参照字体（提供字形） |
| `models/` | 预训练模型 `zi2zi-JiT-B-16.pth` |
| `data/` | 自动生成的数据集 |
| `outputs/` | 训练产物 + 推理 PNG + 补集 PNG |
| `outputs/` | 最终打包 zip |

**环境（AutoDL）：** 控制台选 **PyTorch 2.5.x 镜像**（自带 torch+CUDA，开箱即用）；若用 Miniconda 基础镜像，Cell 1 会自动创建 Python 3.10 环境，装完后按提示切换 kernel 重跑即可。

> ⚡ 本版已按 **V100 32G / 6 核** 预设：`AUTO_TUNE` 用 probe 实测显存自动定 batch（32G 下约 100+，不会 OOM）；数据加载进程已限制为 6（与核数匹配）。

---

## 🧭 建议路线：先小规模验证，再全量补字

**阶段一 · 快速验证（半天内看效果）：**

1. `CHARSET` 先用 `gb2312`（6763 字）、`TRAIN_CHARS_PER_FONT=500`（先小样本训练）
2. 训练完先跑 Cell 4，看测试集生成效果
3. 补集先小量：`MISSING_NUM_IMAGES=100`、`MISSING_PAIRWISE=""`（不写对比图，省一半 I/O、显著提速）
4. 效果满意 → 进入阶段二；笔画模糊/错字多 → 按 Cell 0 参数表"效果不好怎么调"调整

**阶段二 · 全量补字：**

1. `CHARSET="gbk"`（GBK 全量 20,902 字，含生僻字），并同步改下面 3 个参数
2. `TRAIN_CHARS_PER_FONT` 提到 ≥ 字符集大小（如 20902）、`MAX_CHARS_PER_FONT=None`
3. 重新跑 Cell 2（重新生成 GBK 数据集，约 10~30 分钟）→ Cell 3 重训 → Cell 5 全量补集（`MISSING_NUM_IMAGES=None`）

**切到 GBK 要改哪 3 个参数、有什么用：**

| 参数 | 改法 | 作用 |
|---|---|---|
| `CHARSET` | `"gb2312"` → `"gbk"` | 字符范围从 6,763 扩到 20,902（含生僻字），补集能覆盖目标字体缺的所有 GBK 字 |
| `NUM_CHARS` | `20000` → `30000` | 字符嵌入上界必须 ≥ 实际字符数，否则训练报错/截断 |
| `MAX_CHARS_PER_FONT` | `200` → `None` | 默认只让 200 字参与训练；补全场景要让抽出的全部字都进训练集 |

代价：数据准备 10~30 分钟、训练时间明显变长（每轮处理的字多）。


## Cell 0 · 参数预设（训练新字体只改这一个 Cell）

> 只列出**需要关注**的参数：🔴 **必改** = 每次换字体都要改；其余是效果不佳时才调。没列出的参数保持默认即可。

| 参数 | 作用 | 建议 |
|---|---|---|
| **运行开关** | | |
| `DO_DATA_PREP` / `DO_TRAIN` / `DO_GENERATE` / `DO_MISSING_GEN` / `DO_EXPORT` | 分别控制数据准备/训练/推理/补集/导出 | 首次全 `True`；之后可跳过已完成步骤（设 `False`） |
| **🔴 必改：导入素材** | | |
| `TARGET_FONTS` | **要学的新字体**（放 `fonts/`） | 🔴 **必改**：改成你的字体文件名，如 `["fonts/zhufengti.TTF"]`（可多个=多字体） |
| `SOURCE_FONT` | 参照字体，提供"源字形" | 留空自动挑；推荐显式填 `fonts/jigmo/jigmo.ttf` |
| `BASE_CHECKPOINT` | 预训练模型 | 🔴 **必改**：与 `MODEL`/`CFG` 配套——B 模型 → `models/zi2zi-JiT-B-16.pth` + `JiT-B/16` + `CFG=2.6`；L 模型 → 对应 `JiT-L/16` + `CFG=2.4` |
| **数据准备（影响补集效果）** | | |
| `CHARSET` | 字符集 | `gb2312`(6763) 起步够用；要补生僻字 → `"gbk"`(20902)，并同步 `NUM_CHARS=30000`、`MAX_CHARS_PER_FONT=None`（详见上方路线） |
| `TRAIN_CHARS_PER_FONT` | 每字体抽多少字进训练集 | 补集效果最佳 → ≥ 字符集大小（如 500 → 6763）；小规模验证可保持小值 |
| `MAX_CHARS_PER_FONT` | 每字体实际训练字数上限 | 补集 → `None`（全部参与训练）；默认 200 只够试跑 |
| **LoRA 训练（风格相关）** | | |
| `LORA_R` | LoRA 容量 | 风格化明显（行书/草书/手写）→ 64；规整（黑/宋/楷）→ 32 默认 |
| `EPOCHS` | 训练轮数 | 风格化明显 → 300+；规整 → 200 默认足够 |
| `NOISE_SCALE` | 扩散噪声缩放（**训练期固化**进 checkpoint，生成阶段不能覆盖） | **笔画乱/错字多 → 0.8；笔画模糊/缺细节 → 1.2**；默认 1.0 |
| `CFG` | 推理引导强度（越大风格越浓，过大会糊） | 风格化明显 → 3.5~4.0；默认 2.6（L 模型 2.4） |
| **推理生成（Cell 4）** | | |
| `GENERATE_CFG` / `GENERATE_SAMPLING_METHOD` / `GENERATE_NUM_SAMPLING_STEPS` | 效果不好时调 | 按"加大 CFG → 换 `heun` → 加步数"顺序试 |
| `GENERATE_PAIRWISE` | 是否出"源字形\|生成结果"对比图 | `None`=沿用训练默认；非 None 时额外写 `compare/` |
| **缺失字补集（Cell 5）** | | |
| `DO_MISSING_GEN` | 补集总开关 | 需要补缺字时保持 `True` |
| `MISSING_NUM_IMAGES` | 补几张图 | `None`=全部缺失字（gbk 可数千字）；**先设 100 看效果**，满意再全量 |
| `MISSING_PAIRWISE` | 对比图开关 | `"src_gen"`（默认）= 源字形\|生成结果，写盘翻倍；`""`=只出生成图，省一半 I/O、显著提速。注意"源字形"来自**参照字体**（缺失字目标字体根本没有）；`target_gen` 在补集场景**不可用** |
| `MISSING_BATCH_SIZE` | 补集 batch | 128（V100 默认即可，已注释"可调大"） |
| **导出** | | |
| `EXPORT_PREFIX` / `EXPORT_PACK_MISSING` / `EXPORT_PACK_GENERATED` / `EXPORT_PACK_CHECKPOINT` | zip 名前缀 / 是否打包补字图 / 测试推理图 / 训练权重 | 默认 `missing/generated=True`、`checkpoint=False`（拼字体只需补字图） |

**效果不好怎么调（速查）：**

| 现象 | 改法 |
|---|---|
| 笔画模糊 / 缺细节 | 训练前 `NOISE_SCALE` 1.0 → 1.2（需重训）；或推理时加大 CFG、换 `heun`、加步数 |
| 笔画乱 / 错字多 | 训练前 `NOISE_SCALE` 1.0 → 0.8（需重训）；或 `LORA_R` 加大、`EPOCHS` 加多 |
| 风格不够浓 | `CFG` 2.6 → 3.5~4.0；风格化字体配 `LORA_R=64` |
| 训练时间太长 | 换小字符集（gb2312）、减小 `TRAIN_CHARS_PER_FONT`、减小 `EPOCHS` |
| 想跑得快 / 显存紧 | 保持 `AUTO_TUNE=True`（probe 实测自动定 batch），不要手动把 batch 调大 |

> 💡 训练 lr（`BLR=8e-4`）固定、不随 batch 缩放：batch 被 probe 调大后若发现收敛变慢，可把 `BLR` 上调约 1.5~2 倍。


In [ ]:
# ============================================================
# Cell 0 · 参数预设（训练新字体只需修改本 Cell）
# 每个变量都加了简注；详细说明见上方 markdown 表
# ============================================================
import os

# 修复 AutoDL libgomp 报错：平台可能把 OMP_NUM_THREADS 设成空/非法值，
# 导致 Cell 2/5 的子进程一启动就崩（libgomp: Invalid value...）。
# 这里强制设成合法值（CPU 核数）；它只控制 CPU 并行线程数，不影响生成结果。
os.environ['OMP_NUM_THREADS'] = str(max(1, os.cpu_count() or 4))

# ---------- 运行开关（本 Cell 跑哪些步骤） ----------
DO_DATA_PREP   = True     # 重新生成数据集（首次/换字体=True；已有数据集可设 False 跳过）
DO_TRAIN       = True     # 训练 LoRA 模型（生成缺失字前必须先训练）
DO_GENERATE    = True     # 生成"指定字符"推理 PNG
DO_MISSING_GEN = True     # 生成"缺失字补集"（按 CHARSET 补齐目标字体缺的字）
DO_EXPORT      = True     # 打包导出 zip（可直接下载）

# ---------- 目录（素材放项目内 fonts/、models/，本 Cell 会自动创建） ----------
FONTS_DIR       = os.path.join(os.path.abspath(""), "fonts")    # 字体目录：上传目标字体 + 参照字体
MODELS_DIR      = os.path.join(os.path.abspath(""), "models")   # 模型目录：上传预训练模型
DATA_DIR        = os.path.join(os.path.abspath(""), "data")     # 数据集目录（自动生成）
OUTPUTS_DIR     = os.path.join(os.path.abspath(""), "outputs")  # 训练/推理产物目录（自动生成）
EXPORTS_DIR     = os.path.join(os.path.abspath(""), "exports")  # 旧版 zip 导出目录（Cell 6 已改为 outputs，保留兼容）

# ---------- 导入（素材） ----------
SOURCE_FONT     = ""                                 # 参照字体：提供"源字形"内容，必须要有
                                                     # （推荐填 fonts/jigmo/jigmo.ttf；
                                                     #  留空=自动发现 fonts/jigmo/ 或 fonts/ 顶层非目标字体）
TARGET_FONTS    = ["fonts/MyFont.ttf"]         # <-- 必改：改成你的字体文件名，如 ["fonts/zhufengti.TTF"]
BASE_CHECKPOINT = "models/zi2zi-JiT-B-16.pth"        # 预训练模型（B/L 二选一，文件名须与 MODEL 配套）

# ---------- 数据准备 ----------
CHARSET               = "gbk2312"   # 字符集：gb2312(6763)/gbk(20902)/big5/jisx0208/ksx1001
TRAIN_CHARS_PER_FONT  = 500        # 每字体抽多少个字进训练集（补集效果最佳 >= 字符集大小=全部交集字）
TEST_CHARS_PER_FONT   = 8          # 每字体抽多少个"训练未见字"进测试集（交集不足时自动降级为训练字展示）
RESOLUTION            = 256        # 渲染分辨率（必须 = IMG_SIZE）
TRAIN_SEED            = 42         # 训练集抽样种子（复现用）
TEST_SEED             = 99999      # 测试集抽样种子（复现用）
NUM_WORKERS_DATA_PREP = 4          # 数据生成并行进程数

# ---------- LoRA 训练 ----------
MODEL              = "JiT-B/16"   # 模型架构（必须与预训练模型配套：JiT-B/16 或 JiT-L/16）
IMG_SIZE           = 256          # 训练分辨率（必须与预训练模型一致）
NUM_FONTS          = 1000         # 字体嵌入维度（必须 = 预训练模型，改了会加载失败）
NUM_CHARS          = 20000        # 字符嵌入上界（>= 数据集字符数即可；真 GBK 请用 30000）
MAX_CHARS_PER_FONT = 200          # 每字体实际参与训练的字数上限（None=全部；补集建议 None）
LORA_R             = 32           # LoRA 秩：容量（风格化明显 -> 64）
LORA_ALPHA         = 32           # LoRA alpha：缩放（一般 = LORA_R）
LORA_TARGETS       = "qkv,proj,w12,w3"   # LoRA 作用的层（qkv/proj=注意力，w12/w3=FFN）
LORA_DROPOUT       = 0.0          # LoRA dropout（防过拟合）
PROJ_DROPOUT       = 0.1          # 投影层 dropout
EPOCHS             = 200          # 训练轮数（风格化明显 -> 300+）
BLR                = 8e-4         # 基础学习率
MIN_LR             = 1e-6         # 最低学习率（训练后期）
WARMUP_EPOCHS      = 1            # 学习率预热轮数
SAVE_LAST_FREQ     = 10           # 每 N 轮保存一次 last.pt
SEED               = 42           # 训练随机种子（复现用）
P_MEAN             = -0.8         # 噪声调度均值（EDM 参数，一般不调）
P_STD              = 0.8          # 噪声调度标准差（EDM 参数，一般不调）
NOISE_SCALE        = 1.0          # 噪声缩放（笔画乱/错字多 -> 0.8；太模糊 -> 1.2）
CFG                = 2.6          # 推理引导强度（风格化明显 -> 3.5~4.0）
ONLINE_EVAL        = True         # 训练过程中在线评估
EVAL_STEP_FOLDERS  = True         # 评估图按 step 分文件夹保存
EVAL_FREQ          = 10           # 每 N 轮评估一次
NUM_IMAGES         = 6            # 每字体评估生成几张图
BATCH_SIZE         = 16           # 训练 batch（AUTO_TUNE=True 时会被自动覆盖）
GEN_BSZ            = 16           # 评估 batch（AUTO_TUNE=True 时会被自动覆盖）

# ---------- 硬件自动调优（推算 batch/workers；AutoDL 版默认按「云平台独占 GPU」激进化） ----------
AUTO_TUNE           = True        # 自动测显存推算 BATCH_SIZE/GEN_BSZ/NUM_WORKERS
TUNE_METHOD         = "probe"     # probe=实测 / table=标定表（AutoDL 强烈建议 probe）
TUNE_RESERVE        = 0.92        # 显存预留比例（云平台独占 GPU 可激进 0.92~0.95；本地/有桌面占显存则回 0.85）
TUNE_MAX_BATCH      = 256         # 推算出的训练 batch 上限（24G+ 卡设 256 收益大；16G 卡 128 足够）
TUNE_MAX_GEN_BSZ    = 128         # 推理 batch 上限（只前向不吃显存，可大于训练 batch）
TUNE_NUM_WORKERS_CAP = 6          # 数据加载进程上限（V100 实例 6 核，必须 <= 核数，否则线程争抢拖慢）

# ---------- 推理生成（None=沿用训练时配置） ----------
GENERATE_NUM_IMAGES         = None    # 每字生成几张图（None=1）
GENERATE_BATCH_SIZE         = 64      # 推理 batch 大小
GENERATE_CFG                = None    # 推理引导强度（None=沿用 CFG）
GENERATE_SAMPLING_METHOD    = None    # 采样方法（None=沿用训练默认）
GENERATE_NUM_SAMPLING_STEPS = None    # 采样步数（None=沿用训练默认）
GENERATE_PAIRWISE           = None    # 是否生成"源字形|生成结果"对比图（None=沿用训练默认）

# ---------- 缺失字补集 ----------
MISSING_CHARSET         = None           # 补集用哪个字符集（None=沿用 CHARSET）
MISSING_BATCH_SIZE      = 128            # 补集 batch 大小（V100 可调大，减少 batch 切换开销）
MISSING_CFG             = None           # 补集引导强度（None=沿用 CFG）
MISSING_SAMPLING_METHOD = None           # 补集采样方法（None=沿用训练默认）
MISSING_NUM_SAMPLING_STEPS = None        # 补集采样步数（None=沿用训练默认）
MISSING_NUM_IMAGES      = None           # 补几张图（None=全部缺失字）
MISSING_PAIRWISE        = ""      # 输出"源字形|生成结果"对比图（""=只出生成图，省一半写盘、显著提速）
MISSING_REF_CHARS       = ""             # 逗号分隔的样式参考字（留空自动挑）

# ---------- 导出 ----------
# 导出 zip 文件名前缀；最终 zip 名 = <前缀>_<字体>_<时间戳>.zip
EXPORT_PREFIX = "zi2zi_jit"

# 打包范围开关（True=打进 zip，False=跳过）。默认只打补字图和测试推理图，
# 不打权重（权重较大、且拼字体用不到，需要时再设 True）。
EXPORT_PACK_MISSING    = True   # 打包 missing_chars/ 下的补字 PNG（拼字体用）
EXPORT_PACK_GENERATED  = True   # 打包 generated_chars/ 下的测试推理 PNG（效果对比用）
EXPORT_PACK_CHECKPOINT = False  # 打包训练权重 checkpoint-last.pth

# ---------- 派生路径（自动计算，不用改） ----------
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]  # 由目标字体文件名生成的标签
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG)    # 本字体数据集目录
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")  # 训练图像目录
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")  # 测试集（推理用）
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)   # 训练产物目录
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")    # 推理 PNG 目录
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")  # 补集 PNG 目录


## Cell 1 · 环境初始化 + 项目根目录识别 + 硬件自动调优

**本 Cell 做什么（无需修改，直接运行）：**

1. 自动定位项目根目录（识别 marker 文件，不依赖文件夹名字/位置）
2. 检查依赖：缺 torch 自动装；Miniconda 基础镜像（Python 3.7）会自动创建 Python 3.10 环境并注册 kernel，**完成后需手动切换 kernel 后重跑本 Cell 起**
3. 导入检查：目标字体、源字体、预训练模型是否都在（缺什么会报错提示）
4. 配置自检：`MODEL`↔预训练模型是否配套、`CHARSET`/`NUM_CHARS`/`MAX_CHARS_PER_FONT` 是否合理、已有数据缓存与当前 `CHARSET` 是否一致
5. 硬件自动调优：probe 实测显存 → 推算 `BATCH_SIZE`/`GEN_BSZ`/`NUM_WORKERS` 并打印
6. 预下载 InceptionV3 权重（FID/IS 评估用）到数据盘缓存（`fid_stats/.torch_cache/`，已存在则跳过；镜像都失败时会打印手动下载命令）

**产物：** 输出 `PROJECT_ROOT` 与调优后的 `BATCH_SIZE`、`GEN_BSZ`、`NUM_WORKERS`。

> 若自动识别失败（报找不到文件），把 Cell 1 里 `PROJECT_ROOT` 改成项目文件夹绝对路径再跑一次。


In [ ]:
# ============================================================
# Cell 1 · 环境初始化 + 导入检查 + 硬件自动调优（一般不用改）
# ============================================================
import os, sys, glob, shutil, subprocess, datetime, zipfile, io, base64

# 1) 项目根目录 = 项目文件夹（自动识别，不依赖文件夹名字/位置/云环境）
def _find_project_root():
    markers = ("zi2zi_jit_cloudstudio.ipynb", "zi2zi_jit_local.ipynb", "zi2zi_jit_autodl.ipynb",
               "lora_single_gpu_finetune_jit.py", "config_font.py")
    _block = {"windows", "program files", "program files (x86)", "system32",
              "appdata", "programdata", "recovery", "system volume information",
              "$recycle.bin", "perflogs", "msocache", "intel", "amd", "nvidia",
              "documents and settings", "node_modules", ".git", ".venv", "venv",
              "site-packages", "__pycache__", ".idea", ".vscode"}

    def _is_root(p):
        return os.path.isdir(p) and any(os.path.exists(os.path.join(p, m)) for m in markers)

    # a) 当前工作目录（本地 Jupyter / VS Code 打开本 notebook 时通常就是项目根）
    cwd = os.path.abspath("")
    if _is_root(cwd):
        return os.path.normpath(cwd)

    # b) 向上逐级搜索祖先目录：项目文件夹位于工作目录上方时必然命中
    cur = cwd
    while True:
        parent = os.path.dirname(cur)
        if parent == cur:  # 已到盘符根（C:\ 或 /）
            break
        if _is_root(parent):
            return os.path.normpath(parent)
        cur = parent

    # c) 云环境约定目录（仅 Linux：腾讯云 Cloud Studio / Colab 等，兼容原版场景）
    if sys.platform.startswith("linux"):
        for r in ("/workspace", "/content", "/home"):
            if os.path.isdir(r):
                if _is_root(r):
                    return os.path.normpath(r)
                try:
                    for d in sorted(os.listdir(r)):
                        p = os.path.join(r, d)
                        if os.path.isdir(p) and _is_root(p):
                            return os.path.normpath(p)
                except OSError:
                    pass

    # d) 向下受控搜索（兜底：VS Code 打开了项目文件夹的上级目录，内核工作目录在项目上方）
    def _scan_down(root, depth):
        if depth <= 0:
            return None
        try:
            entries = sorted(os.listdir(root))
        except OSError:
            return None
        for e in entries:
            p = os.path.join(root, e)
            if os.path.isdir(p) and not e.startswith(".") and e.lower() not in _block and _is_root(p):
                return p
        for e in entries:
            p = os.path.join(root, e)
            if os.path.isdir(p) and not e.startswith(".") and e.lower() not in _block:
                r = _scan_down(p, depth - 1)
                if r is not None:
                    return r
        return None

    hit = _scan_down(cwd, 4)
    if hit is not None:
        return os.path.normpath(hit)

    # e) 兜底：用当前工作目录并提示手动设置
    print("[Cell1] 警告：未能自动识别项目根目录，使用当前工作目录；"
          "若后续报找不到文件，请手动把 PROJECT_ROOT 改为项目文件夹绝对路径")
    return os.path.normpath(cwd)

PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print("[Cell1] 项目根目录:", PROJECT_ROOT)

# 2) 依赖检查（版本自适应：AutoDL Miniconda 镜像可能是 Python 3.7，装不了 torch 2.x，会自动建 3.10 环境）
import platform as _platform
import importlib.util as _ilu

_TORCH_VER, _TV_VER = "2.5.1", "0.20.1"
_BASE_DEPS = ["numpy", "opencv-python", "timm", "tensorboard", "scipy", "einops",
              "gdown", "fonttools", "Pillow", "pytorch-msssim", "lpips",
              "torch-fidelity", "tqdm", "matplotlib"]
_MODULE_MAP = {"opencv-python": "cv2", "fonttools": "fontTools", "Pillow": "PIL",
               "pytorch-msssim": "pytorch_msssim", "torch-fidelity": "torch_fidelity"}

def _pip(py, *pkgs):
    print("  $", py, "-m pip install", " ".join(pkgs))
    subprocess.run([py, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([py, "-m", "pip", "install", *pkgs], check=True)

def _missing_modules():
    miss = []
    for pkg in _BASE_DEPS + ["torchvision"]:
        mod = _MODULE_MAP.get(pkg, pkg)
        if _ilu.find_spec(mod) is None:
            miss.append(pkg)
    return miss

print("[Cell1] Python:", _platform.python_version(), "| exe:", sys.executable)
try:
    import torch  # noqa
    TORCH_READY = True
except ImportError:
    TORCH_READY = False

if TORCH_READY:
    # torch 已装（如 AutoDL PyTorch 镜像）：只补装缺的包，不覆盖 torch 版本
    miss = _missing_modules()
    if miss:
        print("[Cell1] 补装缺失依赖:", miss)
        _pip(sys.executable, *miss)
elif sys.version_info[:2] >= (3, 9):
    # Python >= 3.9：直接装 torch 2.5.1（AutoDL GPU 驱动支持 CUDA 12.1）
    print("[Cell1] 安装依赖（首次约需几分钟）...")
    _pip(sys.executable, f"torch=={_TORCH_VER}", f"torchvision=={_TV_VER}", *_BASE_DEPS)
else:
    # Python < 3.9（AutoDL Miniconda 基础镜像 = 3.7）：自动创建 Python 3.10 环境并注册 kernel
    print("[Cell1] 当前 Python", _platform.python_version(),
          "过低，torch 2.x 需要 Python >= 3.9。正在用 conda 创建 Python 3.10 环境...")
    conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
    assert conda, ("[Cell1] 未找到 conda：请改用 AutoDL 的 PyTorch 2.5.x 镜像"
                   "（Python >= 3.9）重建实例，或先安装 miniconda")
    env = "zi2zi"
    subprocess.run([conda, "create", "-y", "-n", env, "python=3.10", "pip"], check=True)
    subprocess.run([conda, "run", "-n", env, "python", "-m", "pip", "install",
                    "--upgrade", "pip"], check=True)
    subprocess.run([conda, "run", "-n", env, "python", "-m", "pip", "install",
                    "ipykernel", "jupyter_client"], check=True)
    subprocess.run([conda, "run", "-n", env, "python", "-m", "pip", "install",
                    f"torch=={_TORCH_VER}", f"torchvision=={_TV_VER}", *_BASE_DEPS], check=True)
    subprocess.run([conda, "run", "-n", env, "python", "-m", "ipykernel",
                    "install", "--user", "--name", env,
                    "--display-name", "Python 3.10 (zi2zi)"], check=True)
    print()
    print("=" * 72)
    print("✅ 已创建 Python 3.10 环境并注册 Jupyter kernel: 'Python 3.10 (zi2zi)'")
    print("👉 下一步：菜单 Kernel → Change Kernel → 选 'Python 3.10 (zi2zi)'")
    print("   切换后 Kernel 会重启，请从 Cell 0 开始重新运行本 Notebook。")
    print("=" * 72)
    raise SystemExit(0)

import torch
print("[Cell1] torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| 设备:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 3) 按 PROJECT_ROOT 重新定位路径（覆盖 Cell 0 中的相对值）
FONTS_DIR   = os.path.join(PROJECT_ROOT, "fonts")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models")
DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
EXPORTS_DIR = os.path.join(PROJECT_ROOT, "exports")
FONT_TAG       = os.path.splitext(os.path.basename(TARGET_FONTS[0]))[0]
DATASET_DIR    = os.path.join(DATA_DIR, FONT_TAG)
TRAIN_DIR      = os.path.join(DATASET_DIR, "train")
TEST_NPZ_PATH  = os.path.join(DATASET_DIR, "test.npz")
OUTPUT_DIR     = os.path.join(OUTPUTS_DIR, FONT_TAG)
GEN_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "generated_chars")
MISSING_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "missing_chars")
for d in (FONTS_DIR, MODELS_DIR, DATA_DIR, OUTPUTS_DIR, EXPORTS_DIR):
    os.makedirs(d, exist_ok=True)

# 4) 导入检查：目标字体 / 源字体 / 预训练模型
for t in TARGET_FONTS:
    p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
    assert os.path.exists(p), "[Cell1] 目标字体不存在: %s（请上传到 %s）" % (p, FONTS_DIR)
print("[Cell1] 目标字体 OK:", TARGET_FONTS)

def resolve_source_font():
    if SOURCE_FONT:
        p = SOURCE_FONT if os.path.isabs(SOURCE_FONT) else os.path.join(PROJECT_ROOT, SOURCE_FONT)
        assert os.path.exists(p), "[Cell1] 源字体不存在: " + p
        return p
    target_names = {os.path.basename(t) for t in TARGET_FONTS}
    fonts = []
    for ext in ("*.ttf", "*.otf", "*.ttc"):
        fonts += glob.glob(os.path.join(FONTS_DIR, ext))
    # 也自动搜索 fonts/jigmo/（HanziGen 同款参考字体目录，jigmo.ttf / jigmo2.ttf / jigmo3.ttf）
    jigmo_dir = os.path.join(FONTS_DIR, "jigmo")
    if os.path.isdir(jigmo_dir):
        for ext in ("*.ttf", "*.otf", "*.ttc"):
            fonts += glob.glob(os.path.join(jigmo_dir, ext))
    cand = [f for f in fonts if os.path.basename(f) not in target_names]
    assert cand, "[Cell1] 未找到源字体：请在 fonts/ 放一个参照字体（如 fonts/jigmo/jigmo.ttf），或设置 SOURCE_FONT"
    # jigmo.ttf 覆盖最全，优先；其余按文件名排序
    jigmo_pri = [f for f in cand if os.path.basename(f).lower() in ("jigmo.ttf", "jigmo.otf")]
    return sorted(jigmo_pri or cand)[0]

SOURCE_FONT_PATH = resolve_source_font()
print("[Cell1] 源字体:", SOURCE_FONT_PATH)

ckpt = BASE_CHECKPOINT if os.path.isabs(BASE_CHECKPOINT) else os.path.join(PROJECT_ROOT, BASE_CHECKPOINT)
assert os.path.exists(ckpt), "[Cell1] 预训练模型不存在: %s（请放到 %s）" % (ckpt, MODELS_DIR)
print("[Cell1] 预训练模型:", ckpt)

# 4.5) 配置自检：MODEL<->checkpoint 配套 / 字符集参数 / 数据缓存一致性
import json as _json
import re as _re
from data_processing.charsets import get_charset_codepoints

print("\n[Cell1] ---- 配置自检 ----")
_config_errors = []

def _cok(cond, msg):
    if cond:
        print("  [OK] " + msg)
    else:
        _config_errors.append(msg)
        print("  [ERR] " + msg)

def _cwarn(msg):
    print("  [!!] " + msg)

# a) MODEL <-> BASE_CHECKPOINT 尺寸配套（B-16.pth 配 JiT-B/16，L-16.pth 配 JiT-L/16）
_m_ck = _re.search(r"[_-]([BL])[_-]\d+", os.path.basename(ckpt))
_m_md = _re.search(r"([BL])/", MODEL) if isinstance(MODEL, str) else None
if _m_ck and _m_md:
    _cok(_m_ck.group(1) == _m_md.group(1),
         "checkpoint %s 尺寸=%s 与 MODEL=%s 配套" % (os.path.basename(ckpt), _m_ck.group(1), MODEL))
elif _m_ck and not _m_md:
    _cok(False, "MODEL='%s' 无法解析尺寸（应为 JiT-B/16 或 JiT-L/16）" % MODEL)
else:
    _cwarn("checkpoint 文件名不含 B/L 尺寸标记，跳过配套检查（load_state_dict 严格加载仍会兜底）")

# b) 字符集参数语义
try:
    _cps = get_charset_codepoints(CHARSET)
    _cs = len(_cps)
    print("  [OK] CHARSET=%s 共 %d 字" % (CHARSET, _cs))
except Exception as _e:
    _cps, _cs = None, 0
    _cok(False, "CHARSET='%s' 不受支持: %s" % (CHARSET, _e))

if _cps is not None:
    _cok(NUM_CHARS >= _cs, "NUM_CHARS(%d) >= 字符集大小(%d)" % (NUM_CHARS, _cs))
    if TRAIN_CHARS_PER_FONT > _cs:
        _cwarn("TRAIN_CHARS_PER_FONT(%d) > 字符集大小(%d)，实际会静默取全部 %d 字" % (TRAIN_CHARS_PER_FONT, _cs, _cs))
    if CHARSET == "gbk" and MAX_CHARS_PER_FONT is not None:
        _cwarn("真 GBK 补集建议 MAX_CHARS_PER_FONT=None（当前=%d），否则每字体仅 %d 字参与训练" % (MAX_CHARS_PER_FONT, MAX_CHARS_PER_FONT))
    if TRAIN_CHARS_PER_FONT >= _cs and MAX_CHARS_PER_FONT is not None and MAX_CHARS_PER_FONT < _cs:
        _cwarn("TRAIN_CHARS_PER_FONT 已覆盖全集，但 MAX_CHARS_PER_FONT=%d 会截断到 %d 字" % (MAX_CHARS_PER_FONT, MAX_CHARS_PER_FONT))

# c) 训练配置一致性
_cok(NUM_FONTS >= len(TARGET_FONTS),
     "NUM_FONTS(%d) >= 目标字体数(%d)" % (NUM_FONTS, len(TARGET_FONTS)))
_cok(RESOLUTION == IMG_SIZE, "RESOLUTION(%d) == IMG_SIZE(%d)" % (RESOLUTION, IMG_SIZE))

# d) 已有数据集缓存与 CHARSET 一致性（改了 CHARSET 后旧数据集不会被自动重建）
if os.path.isdir(TRAIN_DIR) or os.path.exists(TEST_NPZ_PATH):
    _old_cs = set()
    for _mf in glob.glob(os.path.join(TRAIN_DIR, "*", "metadata.json")):
        try:
            with open(_mf, encoding="utf-8") as _f:
                _old_cs.add(str(_json.load(_f).get("charset_filter", "")).lower())
        except Exception:
            pass
    if _old_cs:
        _same = _old_cs == {str(CHARSET).lower()}
        if DO_DATA_PREP:
            _cok(_same, "已有数据集 charset=%s 与当前 CHARSET=%s 一致" % (sorted(_old_cs), CHARSET))
        elif not _same:
            _cwarn("已有数据集 charset=%s 与当前 CHARSET=%s 不一致；DO_DATA_PREP=False 将用旧数据集" % (sorted(_old_cs), CHARSET))
        else:
            print("  [OK] 已有数据集 charset=%s 与当前 CHARSET=%s 一致" % (sorted(_old_cs), CHARSET))
    else:
        _cwarn("已有数据集但读不到 metadata.json（可能是旧版本），建议删除 %s 后重跑" % DATASET_DIR)
else:
    print("  [OK] 无已有数据集（首次运行）")

# e) 汇总：有错误则中断本 Cell
if _config_errors:
    print("\n[Cell1] !!! 配置自检发现 %d 个错误，请修正 Cell 0 后重跑本 Cell !!!" % len(_config_errors))
    for _i, _e in enumerate(_config_errors, 1):
        print("  %d) %s" % (_i, _e))
    raise RuntimeError("配置自检未通过：%d 个错误" % len(_config_errors))
print("[Cell1] 配置自检全部通过")
print()

# 5) 硬件自动调优：自动推算 BATCH_SIZE / GEN_BSZ / NUM_WORKERS
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    from util.auto_tune import auto_tune, probe_batch_size  # noqa
    HAS_AUTO_TUNE = True
except ImportError:
    HAS_AUTO_TUNE = False
    def auto_tune(**kw):
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
        per = 0.25 if kw.get("model_name", "JiT-B/16") == "JiT-B/16" else 0.5
        bsz = int(max(total * kw.get("reserve", 0.85) - 1.5, 0) / per) if total else kw.get("batch_size_fallback", 16)
        bsz = max(1, min(bsz, kw.get("max_batch", 128)))
        return {"batch_size": bsz,
                "gen_bsz": max(1, min(bsz, kw.get("max_gen_bsz", 32))),
                "num_workers": min(os.cpu_count() or 1, kw.get("num_workers_cap", 12)),
                "method": "fallback"}

if AUTO_TUNE:
    TUNE = auto_tune(method=TUNE_METHOD, model_name=MODEL, img_size=IMG_SIZE,
                     reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                     max_gen_bsz=TUNE_MAX_GEN_BSZ, num_workers_cap=TUNE_NUM_WORKERS_CAP,
                     batch_size_fallback=BATCH_SIZE)
    BATCH_SIZE = TUNE["batch_size"]
    GEN_BSZ    = TUNE["gen_bsz"]
    NUM_WORKERS = TUNE["num_workers"]
    print("[Cell1] 自动调优完成（%s）-> batch_size:%d gen_bsz:%d num_workers:%d"
          % (TUNE.get("method", "?"), BATCH_SIZE, GEN_BSZ, NUM_WORKERS))
else:
    NUM_WORKERS = min(os.cpu_count() or 1, TUNE_NUM_WORKERS_CAP)
    print("[Cell1] AUTO_TUNE=False，使用固定 batch_size:", BATCH_SIZE)

# ---- 预下载 torch-fidelity 的 InceptionV3 权重（FID/IS 评估用，避免训练时卡在 GitHub 下载） ----
# 缓存目录放到数据盘（项目目录），重装实例不丢；已存在则跳过
_TORCH_CACHE = os.path.join(PROJECT_ROOT, 'fid_stats', '.torch_cache')
os.environ.setdefault('TORCH_HOME', _TORCH_CACHE)
_INCEP = os.path.join(_TORCH_CACHE, 'hub', 'checkpoints', 'weights-inception-2015-12-05-6726825d.pth')
if not os.path.exists(_INCEP):
    import shutil, urllib.request
    _LEGACY = os.path.expanduser('~/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth')
    if os.path.exists(_LEGACY):
        # 复用方案 A：系统默认缓存里已手动放好 -> 复制到数据盘缓存
        os.makedirs(os.path.dirname(_INCEP), exist_ok=True)
        shutil.copy2(_LEGACY, _INCEP)
        print('[Cell1] 从系统缓存 ~/.cache 复用 InceptionV3 权重')
    else:
        os.makedirs(os.path.dirname(_INCEP), exist_ok=True)
        _urls = [
            'https://ghproxy.com/https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth',
            'https://ghfast.top/https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth',
            'https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth',
        ]
        _ok = False
        for _u in _urls:
            try:
                print('[Cell1] 下载 InceptionV3 权重:', _u)
                urllib.request.urlretrieve(_u, _INCEP)
                print('[Cell1] 完成:', os.path.getsize(_INCEP), 'bytes')
                _ok = True
                break
            except Exception as _e:
                print('[Cell1] 下载失败:', _e)
        if not _ok:
            print('[Cell1] 提示：在 AutoDL 终端执行 source /etc/network_turbo 后手动下载到:')
            print('  wget -O %s https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth' % _INCEP)
else:
    print('[Cell1] InceptionV3 权重已存在，跳过下载')
print("[Cell1] 就绪，继续执行 Cell 2（数据准备）")

## Cell 2 · 数据准备（字体 -> `train/` `test/` `test.npz`）

**本 Cell 做什么（无需修改，直接运行）：**

- 调用 `scripts/generate_font_dataset.py` 渲染 `TARGET_FONTS` 的字形，按 `CHARSET` 过滤、抽训练/测试字
- 生成 `data/<字体>/` 下的数据集；已有数据集时自动跳过（**换 `CHARSET` 后必须重跑本 Cell**）

**产物：** `data/<字体>/train/`（训练图）、`data/<字体>/test/`（测试图）、`data/<字体>/test.npz`（推理用）。

> 纯 CPU 渲染，耗时看字符集：gb2312 约几分钟，GBK 全量 2 万字约 10~30 分钟。


In [ ]:
# ============================================================
# Cell 2 · 数据准备（一般不用改）
# ============================================================
if DO_DATA_PREP:
    if os.path.exists(TRAIN_DIR) and os.path.exists(TEST_NPZ_PATH):
        print("[Cell2] 数据集已存在，跳过生成:", DATASET_DIR)
    else:
        staging = os.path.join(DATA_DIR, "_staging_fonts")
        os.makedirs(staging, exist_ok=True)
        for t in TARGET_FONTS:
            p = t if os.path.isabs(t) else os.path.join(PROJECT_ROOT, t)
            shutil.copy2(p, staging)
        cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_font_dataset.py"),
               "--source-font", SOURCE_FONT_PATH,
               "--font-dir", staging,
               "--output-dir", DATASET_DIR,
               "--train-chars-per-font", str(TRAIN_CHARS_PER_FONT),
               "--test-chars-per-font", str(TEST_CHARS_PER_FONT),
               "--resolution", str(RESOLUTION),
               "--charset", CHARSET,
               "--train-seed", str(TRAIN_SEED),
               "--test-seed", str(TEST_SEED),
               "--num-workers", str(NUM_WORKERS_DATA_PREP)]
        print("[Cell2] 命令:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("[Cell2] DO_DATA_PREP=False，跳过")

## Cell 3 · LoRA 训练

**本 Cell 做什么（无需修改，直接运行）：**

- 用 Cell 0 的超参调用 `lora_single_gpu_finetune_jit.py`，全程 GPU，是整套流程最烧机时的一步
- `AUTO_TUNE=True`（probe）会先构建与真实训练**完全一致**的模型（含 LoRA 注入）实测单样本显存，再精调 `batch_size`——V100 32G 下 JiT-B/16 约 100+，**实测保证不会 OOM**；改 `LORA_R`/模型变体/分辨率后也不用再手动猜显存
- 训练中按 `EVAL_FREQ` 在线生成样例图（`outputs/<字体>/`）

**产物：** `outputs/<字体>/checkpoint-last.pth`（LoRA checkpoint，后续 Cell 都用它）。

> 训练轮数、噪声、CFG 等怎么调见 Cell 0 上方表格。


In [ ]:
# ============================================================
# Cell 3 · LoRA 训练（一般不用改）
# ============================================================
if DO_TRAIN:
    import torch
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
        # V100 等老卡（cc<8）：torch.compile 的 Inductor kernel 在 fp16 下数值不稳定
        # （第一个 batch loss 就是 NaN），且首次编译会占满显存（probe 测得 overhead≈28GB，
        # 导致 batch 被压到 4）。这里在 import 模型前禁用编译，与 generate_chars 的 CPU 分支同理。
        print('[Cell3] V100 检测：禁用 torch.compile（修复 loss=NaN 与显存异常占用）')
        torch.compile = lambda fn=None, *a, **k: (lambda inner: inner) if fn is None else fn
    from lora_single_gpu_finetune_jit import get_args_parser, main as lora_main

    argv = [
        "--data_path", TRAIN_DIR,
        "--test_npz_path", TEST_NPZ_PATH,
        "--output_dir", OUTPUT_DIR,
        "--base_checkpoint", BASE_CHECKPOINT,
        "--model", MODEL,
        "--img_size", str(IMG_SIZE),
        "--num_fonts", str(NUM_FONTS),
        "--num_chars", str(NUM_CHARS),
        "--lora_r", str(LORA_R),
        "--lora_alpha", str(LORA_ALPHA),
        "--lora_targets", LORA_TARGETS,
        "--lora_dropout", str(LORA_DROPOUT),
        "--proj_dropout", str(PROJ_DROPOUT),
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--blr", str(BLR),
        "--min_lr", str(MIN_LR),
        "--warmup_epochs", str(WARMUP_EPOCHS),
        "--save_last_freq", str(SAVE_LAST_FREQ),
        "--P_mean", str(P_MEAN),
        "--P_std", str(P_STD),
        "--noise_scale", str(NOISE_SCALE),
        "--cfg", str(CFG),
        "--num_images", str(NUM_IMAGES),
        "--seed", str(SEED),
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    if MAX_CHARS_PER_FONT is not None:
        argv += ["--max_chars_per_font", str(MAX_CHARS_PER_FONT)]

    args = get_args_parser().parse_args(argv)
    if ONLINE_EVAL:
        args.online_eval = True
    if EVAL_STEP_FOLDERS:
        args.eval_step_folders = True
    args.num_workers = NUM_WORKERS

    # probe 实测精调 batch_size（自动反映 LoRA / encoder / 模型变体对显存的影响）
    if AUTO_TUNE and TUNE_METHOD == "probe" and torch.cuda.is_available() and HAS_AUTO_TUNE:
        print("[Cell3] 构建探测模型（与真实训练一致，含 LoRA 注入）实测单样本显存 ...")
        import torch._dynamo
        torch._dynamo.config.cache_size_limit = 128
        from denoiser import Denoiser
        from util.lora_utils import (inject_lora, mark_only_lora_as_trainable,
                                     _is_lora_state_dict, resolve_checkpoint_path)

        probe_model = Denoiser(args)
        probe_model.update_ema = lambda: None
        ckpt_path = resolve_checkpoint_path(args.base_checkpoint)
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
        is_lora = _is_lora_state_dict(sd)
        del ck
        if not is_lora:
            probe_model.load_state_dict(sd, strict=True)
        targets = [t.strip() for t in args.lora_targets.split(",") if t.strip()]
        inject_lora(probe_model.net, targets, r=args.lora_r, alpha=args.lora_alpha, dropout=args.lora_dropout)
        if is_lora:
            probe_model.load_state_dict(sd, strict=True)
        mark_only_lora_as_trainable(probe_model, train_font_emb=True)
        probe_model.to(device)

        safe, per_gb = probe_batch_size(probe_model, device, img_size=IMG_SIZE,
                                        num_fonts=NUM_FONTS, num_chars=NUM_CHARS,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH)
        if safe is not None:
            print("[Cell3] 实测单样本显存: %.3f GB -> batch_size=%d" % (per_gb, safe))
            args.batch_size = safe
            args.gen_bsz = max(1, min(safe, TUNE_MAX_GEN_BSZ))
        del probe_model
        torch.cuda.empty_cache()

    print("[Cell3] 最终 batch_size:", args.batch_size, "| gen_bsz:", args.gen_bsz, "| num_workers:", args.num_workers)
    # ---- 训练前 checkpoint 健康检查（防权重损坏导致的 loss=NaN） ----
    import os
    from util.lora_utils import resolve_checkpoint_path
    _ck_path = resolve_checkpoint_path(args.base_checkpoint) if args.base_checkpoint else None
    if _ck_path and os.path.exists(_ck_path):
        _ck = torch.load(_ck_path, map_location='cpu', weights_only=False)
        _sd = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
        _bad = [k for k, v in _sd.items() if v.is_floating_point() and (torch.isnan(v).any() or torch.isinf(v).any())]
        if _bad:
            raise RuntimeError('[Cell3] base_checkpoint 含 NaN/Inf 权重: %s，请重新下载该模型文件' % _bad[:8])
        _maxw = max((float(v.abs().max()) for k, v in _sd.items() if v.is_floating_point()), default=0.0)
        print('[Cell3] checkpoint 健康检查通过：0 个 NaN/Inf 权重，最大 |权重|=%.3e' % _maxw)
        del _ck, _sd, _bad
    else:
        print('[Cell3] 未找到 base_checkpoint，跳过健康检查')

    lora_main(args)
else:
    print("[Cell3] DO_TRAIN=False，跳过")

## Cell 4 · 推理生成 PNG（测试集字）

**本 Cell 做什么（无需修改，直接运行）：**

- 用 `outputs/<字体>/checkpoint-last.pth` + `data/<字体>/test.npz` 调用 `generate_chars.py`
- 生成 test 集每个字的单字图；仅当 `GENERATE_PAIRWISE` 非 None 时才额外输出"源字形\|生成结果"对比图（写入 `compare/`）
- 扩散采样必须 GPU，量小（仅测试集字），几分钟内完成

**产物：** `outputs/<字体>/generated_chars/`（`generated/` 单字图，可选 `compare/` 对比图）。

> 只针对"目标字体本来就有、且测试集抽中的字"。想补"目标字体缺失的字"请看 **Cell 5**。


In [ ]:
# ============================================================
# Cell 4 · 推理生成（一般不用改）
# ============================================================
if DO_GENERATE:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell4] 未找到 checkpoint: %s" % ckpt

    from generate_chars import get_args_parser, main as gen_main
    args = get_args_parser().parse_args([
        "--checkpoint", ckpt,
        "--test_npz", TEST_NPZ_PATH,
        "--output_dir", GEN_OUTPUT_DIR,
        "--device", "auto",
    ])
    if GENERATE_NUM_IMAGES is not None:
        args.num_images = GENERATE_NUM_IMAGES
    if GENERATE_BATCH_SIZE is not None:
        args.batch_size = GENERATE_BATCH_SIZE
    if GENERATE_CFG is not None:
        args.cfg = GENERATE_CFG
    if GENERATE_SAMPLING_METHOD is not None:
        args.sampling_method = GENERATE_SAMPLING_METHOD
    if GENERATE_NUM_SAMPLING_STEPS is not None:
        args.num_sampling_steps = GENERATE_NUM_SAMPLING_STEPS
    if GENERATE_PAIRWISE is not None:
        args.pairwise = GENERATE_PAIRWISE

    if AUTO_TUNE and torch.cuda.is_available():
        try:
            from util.auto_tune import auto_tune
            args.batch_size = auto_tune(method="table", model_name=MODEL, img_size=IMG_SIZE,
                                        reserve=TUNE_RESERVE, max_batch=TUNE_MAX_BATCH,
                                        max_gen_bsz=TUNE_MAX_GEN_BSZ,
                                        batch_size_fallback=args.batch_size,
                                        verbose=False)["gen_bsz"]
        except Exception:
            pass

    print("[Cell4] 推理生成中 ... batch_size:", args.batch_size)
    gen_main(args)
else:
    print("[Cell4] DO_GENERATE=False，跳过")

## Cell 5 · 缺失字补集生成（缺字补全）

**本 Cell 做什么（无需修改，直接运行）：**

1. 用 fontTools 读目标字体 cmap，算 **CHARSET − 目标字体已覆盖 = 缺失字**
2. 缺失字以**参照字体**的字形为 content 输入生成（所以对比图里的"源字形"来自参照字体）；样式参考来自目标字体自身
3. 用训练好的 checkpoint 逐个生成 PNG，输出 `missing_chars.txt` 清单

**产物：** `outputs/<字体>/missing_chars/`（`generated/U+XXXX.png` 补全字；`compare/` 对比图仅 `MISSING_PAIRWISE="src_gen"` 时生成；`missing_chars.txt` 缺失字清单）

> ⚠️ **为什么它可能比训练还慢**：GPU 计算不多，瓶颈在**每个字写 1~2 个 PNG 的磁盘 I/O**（gbk 全量可达数千字）。提速：`MISSING_PAIRWISE=""`（省一半写盘）、`MISSING_BATCH_SIZE=128`（V100 默认已够）、先 `MISSING_NUM_IMAGES=100` 试效果。
> ✅ **断点续传**：中途中止后重跑本 Cell，会自动跳过 `missing_chars/generated/` 里已生成的 PNG，不重复烧机时。


In [ ]:
# ============================================================
# Cell 5 · 缺失字补集（一般不用改）
# ============================================================
# 保险起见重设一次（只重跑本 Cell 也生效），修复 AutoDL 的 libgomp 报错
os.environ['OMP_NUM_THREADS'] = str(max(1, os.cpu_count() or 4))

if DO_MISSING_GEN:
    ckpt = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
    assert os.path.exists(ckpt), "[Cell5] 未找到 checkpoint: %s" % ckpt

    target = TARGET_FONTS[0]
    target_path = target if os.path.isabs(target) else os.path.join(PROJECT_ROOT, target)

    cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", "generate_missing_chars.py"),
           "--checkpoint", ckpt,
           "--target-font", target_path,
           "--source-font", SOURCE_FONT_PATH,
           "--charset", MISSING_CHARSET or CHARSET,
           "--output-dir", MISSING_OUTPUT_DIR,
           "--batch-size", str(MISSING_BATCH_SIZE or 32),
           "--pairwise", MISSING_PAIRWISE or "none",
           "--seed", str(SEED),
           "--device", "auto"]
    if MISSING_NUM_IMAGES is not None:
        cmd += ["--num-images", str(MISSING_NUM_IMAGES)]
    if MISSING_CFG is not None:
        cmd += ["--cfg", str(MISSING_CFG)]
    if MISSING_SAMPLING_METHOD is not None:
        cmd += ["--sampling-method", MISSING_SAMPLING_METHOD]
    if MISSING_NUM_SAMPLING_STEPS is not None:
        cmd += ["--num-sampling-steps", str(MISSING_NUM_SAMPLING_STEPS)]
    if MISSING_REF_CHARS:
        cmd += ["--ref-chars", MISSING_REF_CHARS]

    print("[Cell5] 命令:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("[Cell5] DO_MISSING_GEN=False，跳过")

## Cell 6 · 导出打包 + 浏览器下载

**本 Cell 做什么（无需修改，直接运行）：**

- 把 Cell 4 的 `generated_chars/` 与 Cell 5 的 `missing_chars/` 下所有 PNG（可选含 checkpoint）打包成 zip
- 保存到 `outputs/`（左侧文件树可下载），并生成浏览器一键下载按钮
- zip 超过 1 GB 时不再生成 base64 链接，请直接从文件树下载

**产物：** `outputs/<前缀>.zip`


In [ ]:
# ============================================================
# Cell 6 · 导出（一般不用改）
# ============================================================
if DO_EXPORT:
    # 按 Cell 0 的三个开关决定打包哪些内容（至少要打一项）
    pack_roots = []
    if EXPORT_PACK_GENERATED and os.path.isdir(GEN_OUTPUT_DIR):
        pack_roots.append(GEN_OUTPUT_DIR)        # 测试推理图 generated_chars/
    if EXPORT_PACK_MISSING and os.path.isdir(MISSING_OUTPUT_DIR):
        pack_roots.append(MISSING_OUTPUT_DIR)     # 补字图 missing_chars/
    if not pack_roots:
        print("[Cell6] 没有选中任何打包范围，跳过（检查 EXPORT_PACK_* 开关）")
    else:
        os.makedirs(OUTPUTS_DIR, exist_ok=True)
        stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_name = "%s_%s_%s.zip" % (EXPORT_PREFIX, FONT_TAG, stamp)
        zip_path = os.path.join(OUTPUTS_DIR, zip_name)

        count = 0
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root_dir in pack_roots:           # 遍历选中的目录，打包 PNG/JPG
                for root, _, files in os.walk(root_dir):
                    for f in sorted(files):
                        if f.lower().endswith((".png", ".jpg", ".jpeg")):
                            full = os.path.join(root, f)
                            zf.write(full, os.path.relpath(full, PROJECT_ROOT))
                            count += 1
            if EXPORT_PACK_CHECKPOINT:            # 可选打包训练权重
                ck = os.path.join(OUTPUT_DIR, "checkpoint-last.pth")
                if os.path.exists(ck):
                    zf.write(ck, "checkpoint/checkpoint-last.pth")
                else:
                    print("[Cell6] 未找到 checkpoint-last.pth，跳过权重")

        size_mb = os.path.getsize(zip_path) / 1024 / 1024
        print("[Cell6] 导出完成: %s（%d 张图片, %.1f MB）" % (zip_path, count, size_mb))

        if size_mb < 1000:
            with open(zip_path, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            from IPython.display import HTML, display
            display(HTML(
                '<a href="data:application/zip;base64,%s" download="%s" '
                'style="font-size:18px;background:#0d6efd;color:#fff;'
                'padding:10px 20px;text-decoration:none;border-radius:6px">'
                '&#128229; 点击下载 %s（%.1f MB）</a>'
                % (b64, zip_name, zip_name, size_mb)))
        else:
            print("[Cell6] zip 较大，请直接在左侧文件树 outputs/ 目录下载: %s" % zip_path)
else:
    print("[Cell6] DO_EXPORT=False，跳过")

## 完成！

**结果一览：**

| 产物 | 位置 |
|---|---|
| 常规推理 PNG | `outputs/<字体>/generated_chars/` |
| 缺失字补集 PNG + 清单 | `outputs/<字体>/missing_chars/` |
| LoRA checkpoint | `outputs/<字体>/checkpoint-last.pth` |
| 导出 zip | `outputs/` |

**调参速查：**

- 生成效果不理想：`GENERATE_CFG` 加大 → `GENERATE_SAMPLING_METHOD="heun"` → 加步数
- 笔画模糊：重训时 `NOISE_SCALE` 1.0 → 1.2；笔画乱/错字多：`NOISE_SCALE` → 0.8
- 风格化明显（行书/草书/手写）：`LORA_R=64`、`EPOCHS=300+`、`CFG=3.5~4.0`
- 补集效果不佳：训练时用完整 CHARSET（`TRAIN_CHARS_PER_FONT` ≥ 字符集大小、`MAX_CHARS_PER_FONT=None`）

> **AutoDL 环境提示**：PyTorch 2.5.x 镜像开箱即用；Miniconda 基础镜像按 Cell 1 提示切换 kernel。
